In [24]:
import pandas as pd
import numpy as np
%run 03_seller_segmentation.ipynb

Avg LTV (repeat): $277.30
Lost repeat customers: 41
Revenue at Risk: $11,467.03
Total Customers: 96,096
Current Repeat Rate: 3.1%
Current Repeat Customers: 2,978
Target Repeat Customers (15%): 14,414
Retention Gap: 11,436
Avg Repeat Customer LTV: $277.30
Revenue Opportunity: $3,171,202.80
996.84
4.185699746751556
H. Net Impact: $-20,091.79
Risky Cash Cow orders: 59,640
Ruined customers: 11,511
Lost LTV from Risky Cash Cows: $3,191,867.20
Revenue from Risky Cash Cows: $9,050,113.00


## Seasonal Patterns

In [25]:
order_level['order_month'] = pd.to_datetime(
    order_level['order_purchase_timestamp'] 
    ).dt.to_period('M')

In [26]:
monthly_trends = order_level.groupby('order_month').agg(
    total_orders = ('order_id','count'),
    delayed_orders = ('is_delayed','sum'),
    avg_review_score = ('review_score','mean')
).reset_index()
monthly_trends['delay_pct'] = (monthly_trends['delayed_orders']/monthly_trends['total_orders'])*100

In [27]:
# Normal months: delay_pct < 5% AND enough orders

normal_mask = (monthly_trends['delay_pct'] < 5) & (monthly_trends['total_orders'] > 500)
normal_data = monthly_trends[normal_mask]

print(f"Normal months count: {len(normal_data)}")
print(normal_data[['order_month', 'total_orders', 'delay_pct']])

Normal months count: 12
   order_month  total_orders  delay_pct
3      2017-01           800   2.750000
4      2017-02          1780   2.752809
5      2017-03          2682   4.325130
7      2017-05          3700   2.864865
8      2017-06          3245   2.927581
9      2017-07          4026   2.682563
10     2017-08          4331   2.816901
11     2017-09          4285   4.247375
12     2017-10          4631   4.038005
18     2018-04          6939   4.409857
20     2018-06          6167   1.151289
21     2018-07          6292   3.305785


In [28]:
# there is high delivery delay in spike months 
# spike months 
spike_months = ['2017-11','2018-02','2018-03']

# convert months in string to compare 
monthly_trends['month_str']  = monthly_trends['order_month'].astype(str)

# normal delay < 5% delay rate
normal_data = monthly_trends[(monthly_trends['delay_pct']< 5)&(monthly_trends['total_orders']>1000)]

# spike months data
spike_data = monthly_trends[monthly_trends['month_str'].isin(spike_months)]

# normal delay rate
normal_rate = normal_data['delayed_orders'].sum()/normal_data['total_orders'].sum()

# spike delay rate
spike_rate = spike_data['delayed_orders'].sum()/spike_data['total_orders'].sum()

# spike months orders 
spike_orders = spike_data['total_orders'].sum()

# excess delays 
excess_delays = spike_orders * (spike_rate - normal_rate)

# LTV impact
ltv_loss = excess_delays * 277.30


In [29]:
print(f"A. Normal delay rate: {normal_rate:.2%}")
print(f"B. Spike delay rate: {spike_rate:.2%}")
print(f"C. Spike month orders: {spike_orders:,}")
print(f"D. Excess delays: {excess_delays:,.0f}")
print(f"E. LTV impact: ${ltv_loss:,.2f}")

A. Normal delay rate: 3.22%
B. Spike delay rate: 14.70%
C. Spike month orders: 21,483
D. Excess delays: 2,466
E. LTV impact: $683,933.72


In [30]:
print("=== FINAL INSIGHTS ===")

# Q1: Retention
print("\n--- Q1: RETENTION ---")
print(f"Total customers: {customer_level['customer_unique_id'].nunique():,}")
print(f"Repeat customers: {len(customer_level[customer_level['total_orders'] > 1]):,}")
print(f"Repeat rate: {len(customer_level[customer_level['total_orders'] > 1]) / len(customer_level) * 100:.1f}%")

# Q2: Sellers
print("\n--- Q2: SELLERS ---")
print(f"Total sellers: {len(seller_level):,}")
print(seller_level['segments'].value_counts())
print(f"Risky Cash Cows LTV loss :$3.17M")

# Q3: Geography
print("\n--- Q3: GEOGRAPHY ---")
print(f"RJ delay rate: 12.7%")
print(f"SP delay rate: 4.2%")
print(f"LTV impact: $112,968")

# Q4: Seasonal
print("\n--- Q4: SEASONAL ---")
print(f"Normal delay rate: 3.22%")
print(f"Spike delay rate: 14.70%")
print(f"LTV impact: $683,934")

print("\n=== DONE ===")

=== FINAL INSIGHTS ===

--- Q1: RETENTION ---
Total customers: 96,096
Repeat customers: 2,997
Repeat rate: 3.1%

--- Q2: SELLERS ---
Total sellers: 3,095
segments
Risky Cash Cows    822
Hidden Gems        819
Deadwight          728
Stars              726
Name: count, dtype: int64
Risky Cash Cows LTV loss :$3.17M

--- Q3: GEOGRAPHY ---
RJ delay rate: 12.7%
SP delay rate: 4.2%
LTV impact: $112,968

--- Q4: SEASONAL ---
Normal delay rate: 3.22%
Spike delay rate: 14.70%
LTV impact: $683,934

=== DONE ===
